In [ ]:
# alle grafieken zijn er even uitgecomment, maar moeten later worden terugegeplaatst

In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import random
import numpy as np 
import math
from haversine import haversine, Unit

from scripts.visualization.map_vessel_routes_v2 import * 

from scripts.PHMM.distributions.AIS_RF_distribution import * 
from scripts.PHMM.distributions.match_distribution import * 

from scripts.PHMM.viterbi_algorithm.states import *
from scripts.PHMM.viterbi_algorithm.forward import *
from scripts.PHMM.viterbi_algorithm.viterbi import *

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
# laat gefilterde cargo data

df_cargo = pd.read_parquet("../../data/processed/cargo_vessels.parquet")
relevant_columns = ["MMSI", "IMO", "BaseDateTime", "LAT", "LON", "SOG", "Heading"]
df_cargo = df_cargo[relevant_columns].reset_index(drop=True)

# converteer snelheden van 'SOG' van knopen naar km/u (1 knots is 1.852 km/u)

df_cargo["SOG_kph"] = df_cargo["SOG"] * 1.852

df_cargo.head(5)

,MMSI,IMO,BaseDateTime,LAT,LON,SOG,Heading,SOG_kph
0,367121040,IMO7634226,2024-01-01 00:00:03,41.54617,-82.72934,0.0,184.0,0.0000
1,212719000,IMO9698238,2024-01-01 00:00:02,34.52975,-120.94588,10.4,322.0,19.2608
2,367050160,IMO7514713,2024-01-01 00:00:03,46.75197,-92.13539,0.0,137.0,0.0000
3,538007479,IMO9347542,2024-01-01 00:00:06,37.68875,-122.69553,6.0,7.0,11.1120
4,477133500,IMO9638434,2024-01-01 00:00:02,38.07199,-124.04890,13.1,327.0,24.2612


In [5]:
# # plot coverage area
# plot_coverage_area()

In [88]:
# plot voorbeeld route
plot_AIS_route(df_cargo, 538009071)

In [7]:
# # plot de verdeling van het aantal observaties per route

# route_lengths = df_cargo.groupby(["MMSI"]).size().reset_index(name = "length")

# bins = [0, 5, 10, 50, 100, 500, 1000, 2000, 5000, 10000, 20000, np.inf]
# labels = ["0-5", "5-10", "10-50", "50-100", "100-500", "500-1000", "1000-2000", "2000-5000", "5000-10000", "10000-20000", ">20000"]

# route_lengths["length_bins"] = pd.cut(route_lengths["length"], bins = bins, labels = labels, right = False)

# bin_counts_len_route = route_lengths["length_bins"].value_counts().sort_index().reset_index()
# bin_counts_len_route.columns = ["length_bins", "count"]

# plt.figure(figsize = (13, 5))
# sns.barplot(data = bin_counts_len_route,  x = "length_bins", y = "count", palette = "viridis")

# for i, row in bin_counts_len_route.iterrows():
#     plt.text(i, row["count"] + 1, str(row["count"]), ha = "center", va = "bottom")

# plt.title("Distribution of route length")
# plt.xlabel("Number of observations per route (binned)")
# plt.ylabel("Number of routes")
# plt.show()


In [8]:
# verwijder routes die minder dan 5 data punten hebben (dit zijn 73 routes)

route_lengths = df_cargo.groupby(["MMSI"]).size().reset_index(name = "length")
valid_MMSI = route_lengths[route_lengths["length"] > 5]["MMSI"]
df_filtered_by_route_length = df_cargo[df_cargo["MMSI"].isin(valid_MMSI)].copy()
df_filtered_by_route_length.head(5)

,MMSI,IMO,BaseDateTime,LAT,LON,SOG,Heading,SOG_kph
0,367121040,IMO7634226,2024-01-01 00:00:03,41.54617,-82.72934,0.0,184.0,0.0000
1,212719000,IMO9698238,2024-01-01 00:00:02,34.52975,-120.94588,10.4,322.0,19.2608
2,367050160,IMO7514713,2024-01-01 00:00:03,46.75197,-92.13539,0.0,137.0,0.0000
3,538007479,IMO9347542,2024-01-01 00:00:06,37.68875,-122.69553,6.0,7.0,11.1120
4,477133500,IMO9638434,2024-01-01 00:00:02,38.07199,-124.04890,13.1,327.0,24.2612


In [9]:
# # check wat de onrealistische route was, die opeens van normale positie naar abnormale positie ging 

# i = df_filtered_by_route_length[
#     (df_filtered_by_route_length["MMSI"] == 538005048) & 
#     (df_filtered_by_route_length["LAT"].between(0, 1))
# ].index[0]

# # plot deze route
# plot_AIS_route(df_filtered_by_route_length, 538005048)

In [10]:
# bereken speed op basis van tijd en afstand verschil tussen i en i-1

df_speed = df_filtered_by_route_length.copy()

df_speed = df_speed.sort_values(["MMSI", "BaseDateTime"]).reset_index(drop=True)
df_speed["TimeDiff"] = df_speed.groupby("MMSI")["BaseDateTime"].diff().dt.total_seconds()

df_speed["prev_LAT"] = df_speed.groupby("MMSI")["LAT"].shift(1)
df_speed["prev_LON"] = df_speed.groupby("MMSI")["LON"].shift(1)

def compute_distance_to_previous_point(row):
    return haversine((row["LAT"], row["LON"]), (row["prev_LAT"], row["prev_LON"]))

df_speed["DistanceDiff"] = df_speed.apply(compute_distance_to_previous_point, axis=1)

df_speed["Speed_kph"] = np.where(
    (df_speed["TimeDiff"] > 0) & (df_speed["DistanceDiff"] > 0),
    (df_speed["DistanceDiff"] / df_speed["TimeDiff"]) * 3600,
    0
)
df_speed.head(5)

,MMSI,IMO,BaseDateTime,LAT,LON,SOG,Heading,SOG_kph,TimeDiff,prev_LAT,prev_LON,DistanceDiff,Speed_kph
0,205717000,IMO9748485,2024-01-01 02:42:35,46.89866,-125.72790,13.2,92.0,24.4464,NaN,NaN,NaN,NaN,0.000000
1,205717000,IMO9748485,2024-01-01 03:09:33,46.89496,-125.59250,13.6,95.0,25.1872,1618.0,46.89866,-125.72790,10.296077,22.908453
2,205717000,IMO9748485,2024-01-01 03:10:44,46.89473,-125.58606,13.6,95.0,25.1872,71.0,46.89496,-125.59250,0.490005,24.845311
3,205717000,IMO9748485,2024-01-01 03:20:54,46.89337,-125.52966,13.8,93.0,25.5576,610.0,46.89473,-125.58606,4.288228,25.307573
4,205717000,IMO9748485,2024-01-01 03:23:14,46.89343,-125.51669,13.7,93.0,25.3724,140.0,46.89337,-125.52966,0.985561,25.343009


In [11]:
# # maak boxplot van de gemiddelde snelheid per schip voor het verwijderen van de outliers

# avg_speeds = df_speed.groupby("MMSI")["Speed_kph"].mean().reset_index()

# sns.boxplot(data = avg_speeds, x = "Speed_kph")
# sns.stripplot(data = avg_speeds, x = "Speed_kph", alpha=0.4, jitter=True)
# plt.title("Gemiddelde snelheid per schip (voor outlier removal)")
# plt.xlabel("Snelheid (km/u)")
# plt.show()

In [12]:
# verwijder outliers 'speed_kph' column (outliers aan de rechterkant, high speeds)

# gebaseerd op deze strategie:
# https://medium.com/@noorfatimaafzalbutt/outliers-detection-and-removal-using-iqr-method-e629aa8089a8 

Q1 = df_speed["Speed_kph"].quantile(0.25)
Q3 = df_speed["Speed_kph"].quantile(0.75)
IQR = Q3 - Q1 

upper_limit = Q3 + 1.5 * IQR
lower_limit = Q1 - 1.5 * IQR

df_speed["is_high_speed_outlier"] = df_speed["Speed_kph"] > upper_limit

high_speed_outliers = (df_speed.loc[df_speed["is_high_speed_outlier"], ["MMSI"]].drop_duplicates())
df_speed_filtered = df_speed[~df_speed["MMSI"].isin(high_speed_outliers["MMSI"])]

# verwijder outliers 'speed' column (outliers aan de linkerkant, low speeds) 
# ZORG DAT JE DE 5 KM DREMPEL AANPAST NAAR EEN VARIABLE DIE GEKOZEN KAN WORDEN

avg_speed_per_ship = df_speed_filtered.groupby("MMSI")["Speed_kph"].mean().reset_index()
valid_avg_speed = avg_speed_per_ship[avg_speed_per_ship["Speed_kph"] >= 5]["MMSI"]
df_speed_filtered = df_speed_filtered[df_speed_filtered["MMSI"].isin(valid_avg_speed)]

In [13]:
# # maak boxplot van de gemiddelde snelheid per schip na outliers removal

# avg_speeds_filtered = df_speed_filtered.groupby("MMSI")["Speed_kph"].mean().reset_index()

# sns.boxplot(data = avg_speeds_filtered, x = "Speed_kph")
# sns.stripplot(data = avg_speeds_filtered, x = "Speed_kph", alpha=0.4, jitter=True)
# plt.title("Gemiddelde snelheid per schip (na outlier removal)")
# plt.xlabel("Snelheid (km/u)")
# plt.show()

In [14]:
# # plot de verdeling van het tijdsverschil tussen twee opeenvolgende punten

# df_time = df_speed_filtered.copy()

# bins = [
#     0, 10, 20, 30, 40, 50, 60, 
#     70, 80, 90, 100, 110, 120, 
#     150, 180, 210, 240, 300, 
#     360, 420, 480, 540, 600, np.inf
# ]
# labels = [
#     "0-10", "10-20", "20-30", "30-40", "40-50", "50-60", # 10 seconds
#     "60-70", "70-80", "80-90", "90-100", "100-110", "110-120", # 10 seconds
#     "120-150", "150-180", "180-210", "210-240", # 30 seconds
#     "240-300", "300-360", "360-420", "420-480", "480-540", "540-600", ">600" # 60 seconds
# ]

# df_time["length_bins"] = pd.cut(df_time["TimeDiff"], bins = bins, labels = labels, right = False)

# bin_counts_time = df_time["length_bins"].value_counts().sort_index().reset_index()
# bin_counts_time.columns = ["length_bins", "count"]

# plt.figure(figsize = (20, 5))
# sns.barplot(data = bin_counts_time,  x = "length_bins", y = "count", palette = "viridis")

# for i, row in bin_counts_time.iterrows():
#     plt.text(i, row["count"] + 1, str(row["count"]), ha = "center", va = "bottom")

# plt.title("Histogram of time difference between consecutive AIS points")
# plt.xlabel("Time difference (in minutes)")
# plt.ylabel("Number of observations")
# plt.xticks(rotation=45)
# plt.show()


In [15]:
# bereken splits routes in continuous routes op basis van grote tijdsgaten

gap_time_threshold = df_speed_filtered["TimeDiff"].quantile(0.99)

def split_into_continuous_segments(df):
    df = df.sort_values("BaseDateTime")
    df["is_new_route"] = df["TimeDiff"] > gap_time_threshold
    df["RouteID"] = df["is_new_route"].cumsum()
    return df 

continuous_segments = []
for mmsi, group in df_speed_filtered.groupby("MMSI"): 
    continuous_segments.append(split_into_continuous_segments(group))
    
df_continuous_routes = pd.concat(continuous_segments).reset_index(drop=True)

# behoud alleen routes met meer dan 5 meetpunten

route_lengths = df_continuous_routes.groupby(["MMSI", "RouteID"]).size().reset_index(name="count")
valid_subroutes = route_lengths[route_lengths["count"] >= 5]
df_continuous_routes = df_continuous_routes.merge(valid_subroutes, on=["MMSI", "RouteID"])

# zorg ervoor dat alle eerste 'TimeDiff' waardes van een route op NaN worden gezet
# het eerste punt heeft namelijk geen vorig punt binnen diezelfde route

first_rows = df_continuous_routes.groupby(["MMSI", "RouteID"]).head(1).index
df_continuous_routes.loc[first_rows, ["TimeDiff", "DistanceDiff", "Speed_kph"]] = np.nan

# herbereken gemiddelde snelheid per 'subroute' (continuous route)
avg_speeds = df_continuous_routes.groupby(["MMSI", "RouteID"])["Speed_kph"].mean().reset_index()
valid_avg_speed = avg_speeds[avg_speeds["Speed_kph"] >= 5]
df_continuous_routes = df_continuous_routes.merge(valid_avg_speed[["MMSI", "RouteID"]], on=["MMSI", "RouteID"])

relevant_columns = ["MMSI", "IMO", "BaseDateTime", "LAT", "LON", "SOG", "Heading", "TimeDiff", "DistanceDiff", "Speed_kph", "RouteID"]
df_continuous_routes = df_continuous_routes[relevant_columns].sort_values(["MMSI", "BaseDateTime"]).reset_index(drop=True)

df_continuous_routes.head(5)

,MMSI,IMO,BaseDateTime,LAT,LON,SOG,Heading,TimeDiff,DistanceDiff,Speed_kph,RouteID
0,205717000,IMO9748485,2024-01-01 03:40:24,46.89339,-125.42151,13.7,93.0,NaN,NaN,NaN,3
1,205717000,IMO9748485,2024-01-01 03:44:15,46.89344,-125.39996,13.8,93.0,231.0,1.637508,25.519605,3
2,205717000,IMO9748485,2024-01-01 03:45:33,46.89350,-125.39274,13.9,94.0,78.0,0.548659,25.322724,3
3,205717000,IMO9748485,2024-01-01 03:46:39,46.89348,-125.38572,13.9,93.0,66.0,0.533426,29.095947,3
4,205717000,IMO9748485,2024-01-01 03:48:20,46.89355,-125.37695,13.9,93.0,101.0,0.666442,23.754352,3


In [16]:
# genereer willekeurige tijdstippen met ~3 uur intervallen voor RF-signalen (Unseenlabs revisit time ≈ 3 uur)
# https://earth.esa.int/eogateway/missions/unseenlabs/description

def generate_random_start_time():
    """
    Generates a random start time between 00:00 and 02:59

    Returns:
        datetime: A random time representing the start time
    """
    start_hour = random.randint(0, 2)
    start_minute = random.randint(0, 59)
    start_time = datetime.strptime(f"{start_hour:02d}:{start_minute:02d}", "%H:%M")
    return start_time 

def generate_synthetic_RF_data():
    """
    Simulates RF revisit timestamps by starting from a randomly generated start time 
    and adding time intervals of approximately 3 to 4 hours

    Returns:
        list: Timestamps representing sythethic RF revisit times
    """
    random.seed(42)
    
    start_time = generate_random_start_time()
    times = [start_time]
    
    while True:
        minutes = random.randint(180, 240)
        seconds = random.randint(0,59)
        next_time = times[-1] + timedelta(minutes = minutes, seconds = seconds)
        
        if next_time.day != start_time.day:
            break
        
        times.append(next_time)
    return times

def generate_error_distance_and_bearing(
    timestamps, 
    mean_distance = 2000, 
    std_dev_distance = 100, 
    bearing_options = [0, 45, 90, 135, 180, 225, 270, 315]
    ):
    """
    Generates an error distance and bearing angle for each RF timestamp

    Args:
        timestamps (list): Simulated RF timestamps
        mean_distance (int): The average error distance in meters between the AIS point and the simulated RF signal, defaults to 2000
        std_dev_distance (int): Indicates how mcuh the generated distances can deviate from the average value. 
                                Given a mean of 2000 and a standard deviation of 100, most generated distances will fall roughly between 1900 and 2100 meters
        bearing_options (list): A list of possible direction (in degrees) that are added as deviations to the original course of the vessel, 
                                defaults to [0, 45, 90, 135, 180, 225, 270, 315]

    Returns:
        dict: timestamp: [error_distance, error_bearing]
    """
    return {
        timestamp: [
            round(np.random.normal(mean_distance, std_dev_distance), 6),
            random.choice(bearing_options)
        ] 
        for timestamp in timestamps
    }

def compute_coordinates(latitude, longitude, heading, error_distance, error_bearing):
    """
    Computes a new geographical coordinate from a base position using a error distance and bearing

    Args:
        latitude (float): Latitude of the reference point
        longitude (float): Longitude of the reference point
        heading (float): Direction of the ship at the reference point
        error_distance (float): Distance in meters to offset from the reference point
        error_bearing (float): Max error to add or subtract from the 'original' heading

    Returns:
        tuple: (latitude, longitude) of the simulated RF signal location
    """
    R = 6371000  

    noise = np.random.uniform(-error_bearing, error_bearing)
    bearing = math.radians((heading + noise) % 360)

    latitude, longitude = math.radians(latitude), math.radians(longitude)
    
    new_latitude = math.asin(
        math.sin(latitude) * math.cos(error_distance / R) + 
        math.cos(latitude) * math.sin(error_distance / R) * math.cos(bearing)
    )
    new_longitude = longitude + math.atan2(
        math.sin(bearing) * math.sin(error_distance / R) * math.cos(latitude),
        math.cos(error_distance / R) - math.sin(latitude) * math.sin(new_latitude)
    )

    new_latitude = math.degrees(new_latitude)
    new_longitude = math.degrees(new_longitude)
    
    return new_latitude, new_longitude

In [17]:
def get_RF_times_for_AIS_seq(RF_timestamps, AIS_seq, time_margin):
    """
    Selects RF timestamps that fall within the time window of an AIS route
    
    Args:
        timestamps (list): Simulated RF timestamps
        AIS_sequence (pd.DataFrame): AIS data
        time_margin (int): Times in minutes to expand the window before and after the AIS sequence

    Returns:
        list: RF timestamps that fall within the expanded AIS window
    """
    start_time = AIS_seq["BaseDateTime"].iloc[0] - timedelta(minutes = time_margin)
    end_time = AIS_seq["BaseDateTime"].iloc[-1] + timedelta(minutes = time_margin)
    unique_dates = AIS_seq['BaseDateTime'].dt.normalize().unique()
    
    return sorted([
        datetime.combine(pd.to_datetime(date).date(), RF_time.time())
        for date in unique_dates for RF_time in RF_timestamps
        if start_time <= datetime.combine(pd.to_datetime(date).date(), RF_time.time()) <= end_time
    ])
    
def generate_training_data(df, time_margin = 5):
    """
    Generates labeled training data by combining AIS trajectories with simulated RF signal observations
    
    For each AIS trajectory, and for every simulated RF timestamp that falls within its extended time window, the nearest AIS point is identified.
    A classification decision is then made based on the time and distance between the RF and AIS point.
        - If the probability of a match (based on time and distance) is higher than a standalone RF point, the RF signal is marked as a match ("M")
        - Otherwise, it is treated as an unmatched RF point ("RF")

    Args:
        df (pd.DataFrame): AIS data
        time_margin (int):  Number of minutes to extend the AIS route (before and after), defaults to 5

    Returns:
        pd.DataFrame: Labeled sequence data with the following columns:
            - UniqueRouteID
            - SubRouteID
            - MMSI
            - Timestamp
            - AIS_Timestamp 
            - RF_Timestamp
            - AIS (coordinates)
            - RF (coordinates)
            - State (one of: "M", "AIS", "RF", "begin", "end")
    """
    RF_revisit_timestamps = generate_synthetic_RF_data()
    train_data = []
    distance_differences = []
    
    for (mmsi, route_id), AIS_seq in df.groupby(["MMSI", "RouteID"]):
        AIS_seq["Matched"] = False
        
        RF_times = get_RF_times_for_AIS_seq(RF_revisit_timestamps, AIS_seq, time_margin)
        RF_erros = generate_error_distance_and_bearing(RF_times)
        
        for RF_datetime in RF_times:       
            # Find closest AIS information 
            closest_time_idx = (AIS_seq["BaseDateTime"] - RF_datetime).abs().idxmin()
            closest_AIS_time = AIS_seq.loc[closest_time_idx, "BaseDateTime"]
            closest_AIS_lat = AIS_seq.loc[closest_time_idx, "LAT"]
            closest_AIS_lon = AIS_seq.loc[closest_time_idx, "LON"]
            closest_AIS_heading = AIS_seq.loc[closest_time_idx, "Heading"]
            
            # Calculate the coordinates for RF signal
            distance, bearing = RF_erros[RF_datetime]
            synthetic_RF_lat, synthetic_RF_lon = compute_coordinates(closest_AIS_lat, closest_AIS_lon, closest_AIS_heading, distance, bearing)
            
            # Calculate the difference in distance and time between AIS and RF point
            distance_diff = haversine((closest_AIS_lat, closest_AIS_lon), (synthetic_RF_lat, synthetic_RF_lon))
            distance_differences.append(distance_diff)
            time_diff = abs((RF_datetime - closest_AIS_time).total_seconds())
            
            # Compute the likelihood that this RF signal is a match with an AIS point, or just a standalone RF signal
            prob_RF = AIS_RF_probability(time_diff, distance_diff)
            prob_match = match_probability(time_diff, distance_diff)
            
            if prob_match > prob_RF:
                state = "M"
                AIS_seq.at[closest_time_idx, "Matched"] = True 
            else:
                state = "RF"

            train_data.append({
                "SubRouteID": route_id,
                "MMSI": mmsi,
                "Timestamp": closest_AIS_time if state == "M" else RF_datetime,
                "AIS_Timestamp": closest_AIS_time if state == "M" else pd.NaT,
                "RF_Timestamp": RF_datetime,
                "AIS": [closest_AIS_lat, closest_AIS_lon] if state == "M" else None,
                "RF": [synthetic_RF_lat, synthetic_RF_lon],
                "State": state
            })
            
        for i, row in AIS_seq[~AIS_seq["Matched"]].iterrows():
            train_data.append({
                "SubRouteID": route_id,
                "MMSI": mmsi,
                "Timestamp": row["BaseDateTime"],
                "AIS_Timestamp": row["BaseDateTime"],
                "RF_Timestamp": pd.NaT,
                "AIS": [row["LAT"], row["LON"]],
                "RF": None,
                "State": "AIS"
            })
        
        time_begin = AIS_seq["BaseDateTime"].min() - timedelta(seconds = 1) 
        time_end = AIS_seq["BaseDateTime"].max() + timedelta(seconds = 1)
        
        for time, state in [(time_begin, "begin"), (time_end, "end")]:
            train_data.append({
                "SubRouteID": route_id,
                "MMSI": mmsi,
                "Timestamp": time,
                "AIS_Timestamp": pd.NaT,
                "RF_Timestamp": pd.NaT,
                "AIS": None,
                "RF": None,
                "State": state
            })  
    
    df_training = pd.DataFrame(train_data).sort_values(["MMSI", "SubRouteID", "Timestamp"])
    df_training.insert(0, "UniqueRouteID", df_training.groupby(["SubRouteID", "MMSI"]).ngroup())    
    return df_training, distance_differences

In [18]:
df_train_set, list_distance_diffs = generate_training_data(df_continuous_routes)

In [32]:
forward_results = pd.read_pickle("../../scripts/test_scripts/alignments_df.pkl")

In [ ]:
forward_results.sort_values(["RFRouteID", "RFSignalID"])

,UniqueRouteID,SubRouteID,MMSI,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
382,1396,3,205717000,2024-01-01 03:40:23,NaT,NaT,None,None,begin
4,1396,3,205717000,2024-01-01 03:40:24,2024-01-01 03:40:24,NaT,"[46.89339, -125.42151]",None,AIS
5,1396,3,205717000,2024-01-01 03:44:15,2024-01-01 03:44:15,NaT,"[46.89344, -125.39996]",None,AIS
6,1396,3,205717000,2024-01-01 03:45:33,2024-01-01 03:45:33,NaT,"[46.8935, -125.39274]",None,AIS
7,1396,3,205717000,2024-01-01 03:46:39,2024-01-01 03:46:39,NaT,"[46.89348, -125.38572]",None,AIS
...,...,...,...,...,...,...,...,...,...
4422281,7321,19,750000076,2024-01-05 03:55:30,2024-01-05 03:55:30,NaT,"[16.36872, -65.5559]",None,AIS
4422282,7321,19,750000076,2024-01-05 03:58:10,2024-01-05 03:58:10,NaT,"[16.36608, -65.55114]",None,AIS
4422283,7321,19,750000076,2024-01-05 04:04:39,2024-01-05 04:04:39,NaT,"[16.35969, -65.53966]",None,AIS
4422284,7321,19,750000076,2024-01-05 04:07:50,2024-01-05 04:07:50,NaT,"[16.35669, -65.53414]",None,AIS


In [ ]:
# alles hierboven is belangrijk 

In [60]:
df_train_set[df_train_set["UniqueRouteID"] == 1][10:20]

,UniqueRouteID,SubRouteID,MMSI,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
5473,1,0,209136000,2024-01-08 22:56:24,2024-01-08 22:56:24,NaT,"[28.53924, -89.69059]",None,AIS
5474,1,0,209136000,2024-01-08 22:57:35,2024-01-08 22:57:35,NaT,"[28.54235, -89.68775]",None,AIS
5475,1,0,209136000,2024-01-08 22:59:35,2024-01-08 22:59:35,NaT,"[28.54758, -89.68319]",None,AIS
5476,1,0,209136000,2024-01-08 23:00:25,2024-01-08 23:00:25,NaT,"[28.54966, -89.68132]",None,AIS
5477,1,0,209136000,2024-01-08 23:02:25,2024-01-08 23:02:25,NaT,"[28.55486, -89.67693]",None,AIS
5478,1,0,209136000,2024-01-08 23:07:44,2024-01-08 23:07:44,NaT,"[28.56874, -89.66455]",None,AIS
5450,1,0,209136000,2024-01-08 23:08:37,NaT,2024-01-08 23:08:37,None,"[28.54970914486886, -89.66553563299703]",RF
5479,1,0,209136000,2024-01-08 23:12:55,2024-01-08 23:12:55,NaT,"[28.58203, -89.65335]",None,AIS
5480,1,0,209136000,2024-01-08 23:14:14,2024-01-08 23:14:14,NaT,"[28.58532, -89.65054]",None,AIS
5481,1,0,209136000,2024-01-08 23:15:45,2024-01-08 23:15:45,NaT,"[28.58914, -89.64729]",None,AIS


In [86]:
time_marge = timedelta(minutes = 10)
distance_threshold = np.percentile(list_distance_diffs, 99)

df_RF = df_train_set[df_train_set["RF"].notna()].copy()
df_RF["RFSignalID"] = df_RF.groupby("UniqueRouteID").cumcount() + 1
RF_signal = df_RF[(df_RF["UniqueRouteID"] == 1) & (df_RF["RFSignalID"] == 1)].iloc[0]

RF_datetime = RF_signal["RF_Timestamp"]
RF_coordinates = RF_signal["RF"]
RF_seq_id = RF_signal["UniqueRouteID"]
RF_signal_id = RF_signal["RFSignalID"]

print(RF_coordinates)

# RF_signal_within_radius = any(haversine(RF_coordinates, coord) <= distance_threshold for coord in AIS_coordinates)

t = df_train_set[df_train_set["UniqueRouteID"] == 1]
AIS_coordinates = [coord for coord in t["AIS"] if coord is not None]
RF_signal_within_radius = any(haversine(RF_coordinates, coord) <= distance_threshold for coord in AIS_coordinates)

# Check if RF signal within time bounds
min_AIS_datetime = t["AIS_Timestamp"].min() - time_marge
max_AIS_datetime = t["AIS_Timestamp"].max() + time_marge
RF_signal_within_time = min_AIS_datetime <= RF_datetime <= max_AIS_datetime

print(RF_signal_within_radius, RF_signal_within_time)

[28.54970914486886, -89.66553563299703]
True True


In [79]:
haversine([28.45622, -89.76579], [28.54970914486886, -89.66553563299703])

14.284269067590678

In [63]:
t

,UniqueRouteID,SubRouteID,MMSI,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
5473,1,0,209136000,2024-01-08 22:56:24,2024-01-08 22:56:24,NaT,"[28.53924, -89.69059]",None,AIS
5474,1,0,209136000,2024-01-08 22:57:35,2024-01-08 22:57:35,NaT,"[28.54235, -89.68775]",None,AIS
5475,1,0,209136000,2024-01-08 22:59:35,2024-01-08 22:59:35,NaT,"[28.54758, -89.68319]",None,AIS
5476,1,0,209136000,2024-01-08 23:00:25,2024-01-08 23:00:25,NaT,"[28.54966, -89.68132]",None,AIS
5477,1,0,209136000,2024-01-08 23:02:25,2024-01-08 23:02:25,NaT,"[28.55486, -89.67693]",None,AIS
5478,1,0,209136000,2024-01-08 23:07:44,2024-01-08 23:07:44,NaT,"[28.56874, -89.66455]",None,AIS
5450,1,0,209136000,2024-01-08 23:08:37,NaT,2024-01-08 23:08:37,None,"[28.54970914486886, -89.66553563299703]",RF
5479,1,0,209136000,2024-01-08 23:12:55,2024-01-08 23:12:55,NaT,"[28.58203, -89.65335]",None,AIS
5480,1,0,209136000,2024-01-08 23:14:14,2024-01-08 23:14:14,NaT,"[28.58532, -89.65054]",None,AIS
5481,1,0,209136000,2024-01-08 23:15:45,2024-01-08 23:15:45,NaT,"[28.58914, -89.64729]",None,AIS


In [53]:
# filter train data op waar RF-data aanwezig is
df_RF = df_train_set[df_train_set["RF"].notna()].copy()
df_RF["RFSignalID"] = df_RF.groupby("UniqueRouteID").cumcount() + 1

df_RF.sort_values(["UniqueRouteID", "RFSignalID"])

,UniqueRouteID,SubRouteID,MMSI,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,RFSignalID
5450,1,0,209136000,2024-01-08 23:08:37,NaT,2024-01-08 23:08:37,None,"[28.54970914486886, -89.66553563299703]",RF,1
5451,1,0,209136000,2024-01-09 02:07:00,NaT,2024-01-09 02:07:00,None,"[28.880212839110825, -89.39827545135141]",RF,2
5452,1,0,209136000,2024-01-09 05:08:47,NaT,2024-01-09 05:08:47,None,"[29.21337692363925, -89.30115690642349]",RF,3
5453,1,0,209136000,2024-01-09 08:26:02,NaT,2024-01-09 08:26:02,None,"[29.244333455170963, -89.28292462563665]",RF,4
5454,1,0,209136000,2024-01-09 11:39:21,2024-01-09 11:39:21,2024-01-09 11:40:10,"[29.22886, -89.29829]","[29.242896828683154, -89.308218026839]",M,5
...,...,...,...,...,...,...,...,...,...,...
2278081,16717,539,368099310,2024-01-29 05:08:47,NaT,2024-01-29 05:08:47,None,"[28.574595758570528, -89.15385472830413]",RF,3
1782951,16720,550,366014000,2024-01-25 05:08:47,NaT,2024-01-25 05:08:47,None,"[28.687357525668737, -88.28068633125442]",RF,1
1782986,16723,556,366014000,2024-01-25 08:26:02,NaT,2024-01-25 08:26:02,None,"[28.763690241954983, -88.40775444213239]",RF,1
1782987,16723,556,366014000,2024-01-25 11:40:10,NaT,2024-01-25 11:40:10,None,"[28.736537629299672, -88.36200190201188]",RF,2


In [37]:
# run forward algorithm:
states = [ BeginState(), AISState(), RFState(), MState(), EndState()]
forward_model = PHMM_forward(states, EndState())

# filter train data op waar RF-data aanwezig is
df_RF = df_train_set[df_train_set["RF"].notna()].copy()
df_RF["RFSignalID"] = df_RF.groupby("UniqueRouteID").cumcount() + 1

distance_threshold = np.percentile(list_distance_diffs, 99)
time_marge = timedelta(minutes = 10)
alignment_results = []

for _, RF_signal in df_RF.iterrows():
    RF_datetime = RF_signal["RF_Timestamp"]
    RF_coordinates = RF_signal["RF"]
    RF_seq_id = RF_signal["UniqueRouteID"]
    RF_signal_id = RF_signal["RFSignalID"]
        
    for AIS_seq_id, AIS_seq in df_train_set.groupby("UniqueRouteID"):        
        # Check if RF signal within distance threshold
        AIS_coordinates = [coord for coord in AIS_seq["AIS"] if coord is not None]
        RF_signal_within_radius = any(haversine(RF_coordinates, coord) <= distance_threshold for coord in AIS_coordinates)
        
        # Check if RF signal within time bounds
        min_AIS_datetime = AIS_seq["AIS_Timestamp"].min() - time_marge
        max_AIS_datetime = AIS_seq["AIS_Timestamp"].max() + time_marge
        RF_signal_within_time = min_AIS_datetime <= RF_datetime <= max_AIS_datetime

        if RF_signal_within_radius and RF_signal_within_time:
            # Make sequences 
            AIS_seq = [(datetime, coordinates)
                    for datetime, coordinates in zip(AIS_seq["AIS_Timestamp"], AIS_seq["AIS"])
                    if pd.notna(datetime) and coordinates is not None]
            RF_seq = [(RF_datetime, RF_coordinates)]
            
            is_true_match = RF_seq_id == AIS_seq_id
            
            alignment_results.append({
                "RFSignalID": RF_signal_id,
                "RFRouteID": RF_seq_id,
                "AISRouteID": AIS_seq_id,
                "IsTrueMatch": is_true_match
            })

KeyboardInterrupt: 

In [38]:
alignment_results

[{'RFSignalID': 1, 'RFRouteID': 1396, 'AISRouteID': 1396, 'IsTrueMatch': True},
 {'RFSignalID': 2, 'RFRouteID': 1396, 'AISRouteID': 1396, 'IsTrueMatch': True}]

In [22]:
distance_threshold.round(2)

np.float64(2.23)

In [20]:
alignment_results

[{'RFSignalID': 1,
  'RFRouteID': 1396,
  'AISRouteID': 1396,
  'ForwardScore': np.float64(-15.9000292048),
  'NormalizedForwardScore': np.float64(-0.042063569324867725),
  'IsTrueMatch': True}]

In [ ]:
# stap 2, preselectie van mogelijke AIS sequenties op basis van:
    # tijdsvenster (+- 5 minuten)
    # ruimtelijke afstand (2 km van een van de AIS punten)

# stap 3, forward algorithme draaien op:
    # elke combinatie van het RF signaal + kandidaat AIS sequentie
    # sla deze allemaal op in een dataframe + ook de adjusted forward score (forward / len(AIS_seq))
    
candidates = []
distance_threshold = 2
time_marge = pd.Timedelta(minutes = 10) # deze 10 minuten 

for _, RF_signal in RF_signals_sample.iterrows():
    RF_datetime = RF_signal["RF_Timestamp"]
    RF_coordinates = RF_signal["RF"]
    RF_seq_id = RF_signal["UniqueRouteID"]
    RF_signal_id = RF_signal["RF_index_in_route"]
    
    best_score = -np.inf
    best_candidate = None
    best_adjusted_score = None
    
    for AIS_seq_id, AIS_seq in PHMM_df.groupby("UniqueRouteID"):
        # Check if RF signal within distance threshold
        AIS_coordinates = [coord for coord in AIS_seq["AIS"] if coord is not None]
        RF_signal_within_radius = any(haversine(RF_coordinates, coord) <= distance_threshold for coord in AIS_coordinates)
        
        # Check if RF signal within time bounds
        min_AIS_datetime = AIS_seq["AIS_Timestamp"].min() - time_marge
        max_AIS_datetime = AIS_seq["AIS_Timestamp"].max() + time_marge
        RF_signal_within_time = min_AIS_datetime <= RF_datetime <= max_AIS_datetime

        if RF_signal_within_radius and RF_signal_within_time:
            # Make sequenties 
            AIS_seq = [(datetime, coordinates)
                    for datetime, coordinates in zip(AIS_seq["AIS_Timestamp"], AIS_seq["AIS"])
                    if pd.notna(datetime) and coordinates is not None]
            RF_seq = [(RF_datetime, RF_coordinates)]
        
            # Forward algorithm 
            forward_score = forward_model.forward(AIS_seq, RF_seq)
            adjusted_forward_score = forward_score/len(AIS_seq)
            
            is_true_match = RF_seq_id == AIS_seq_id
            
            candidates.append({
                "RF_signal_ID": RF_signal_id,
                "RF_seq_id": RF_seq_id,
                "AIS_seq_ID": AIS_seq_id,
                "Forward_score": forward_score,
                "Adjusted_Forward_score": adjusted_forward_score, 
                "y_true": is_true_match
            })

In [21]:
from scripts.PHMM.viterbi_algorithm.states import *
from scripts.PHMM.viterbi_algorithm.viterbi import *
from scripts.PHMM.distributions.AIS_RF_distribution import *
import pandas as pd

begin = BeginState()
AIS = AISState()
RF = RFState()
M = MState()
end = EndState()

states = [begin, AIS, RF, M, end]
viterbi_model = PHMM(states, end)

In [22]:
sequence = df_train_set[df_train_set['UniqueRouteID'] == 1]

AIS_sequence = [(time, coord) for time, coord in zip(sequence["AIS_Timestamp"], sequence["AIS"]) 
                if pd.notna(time) and coord is not None]

RF_sequence = [(time, coord) for time, coord in zip(sequence["RF_Timestamp"], sequence["RF"]) 
               if pd.notna(time) and coord is not None]

In [23]:
path, probability, log_probs = viterbi_model.viterbi(AIS_sequence, RF_sequence)

In [24]:
RF_M_indexes_pred_path = [(i, state) for i, state in enumerate(path) if state in {"RF", "M"}]
RF_M_indexes_real_path = [(i, state) for i, state in enumerate(sequence["State"]) if state in {"RF", "M"}]
print(f"The indexes of RF and/or M of predicted path are: {RF_M_indexes_pred_path}, \
      \nThe indexes of RF and/or M of the real path are: {RF_M_indexes_real_path}")

The indexes of RF and/or M of predicted path are: [(16, 'RF'), (159, 'RF'), (313, 'RF'), (379, 'M'), (441, 'RF'), (548, 'RF'), (748, 'RF'), (897, 'M'), (957, 'RF'), (1017, 'M'), (1081, 'RF'), (1146, 'RF'), (1219, 'M'), (1291, 'RF')],       
The indexes of RF and/or M of the real path are: [(15, 'M'), (158, 'RF'), (312, 'RF'), (379, 'M'), (439, 'M'), (546, 'RF'), (746, 'RF'), (895, 'M'), (955, 'RF'), (1015, 'M'), (1079, 'RF'), (1144, 'RF'), (1218, 'M'), (1289, 'RF')]


In [33]:
sequence2 = df_train_set[df_train_set['UniqueRouteID'] == 190]

AIS_sequence2 = [(time, coord) for time, coord in zip(sequence2["AIS_Timestamp"], sequence2["AIS"]) 
                if pd.notna(time) and coord is not None]

RF_sequence2 = [(time, coord) for time, coord in zip(sequence2["RF_Timestamp"], sequence2["RF"]) 
               if pd.notna(time) and coord is not None]

In [34]:
sequence2

,UniqueRouteID,SubRouteID,MMSI,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
1795958,190,0,366893910,2024-01-01 00:01:28,NaT,NaT,None,None,begin
1793962,190,0,366893910,2024-01-01 00:01:29,2024-01-01 00:01:29,NaT,"[43.01629, -87.9017]",None,AIS
1793963,190,0,366893910,2024-01-01 00:04:32,2024-01-01 00:04:32,NaT,"[43.0163, -87.90171]",None,AIS
1793964,190,0,366893910,2024-01-01 00:07:30,2024-01-01 00:07:30,NaT,"[43.01631, -87.90171]",None,AIS
1793965,190,0,366893910,2024-01-01 00:10:31,2024-01-01 00:10:31,NaT,"[43.0163, -87.90169]",None,AIS
...,...,...,...,...,...,...,...,...,...
1795954,190,0,366893910,2024-01-02 23:39:55,2024-01-02 23:39:55,NaT,"[44.5061, -88.02284]",None,AIS
1795955,190,0,366893910,2024-01-02 23:41:05,2024-01-02 23:41:05,NaT,"[44.50697, -88.02252]",None,AIS
1795956,190,0,366893910,2024-01-02 23:42:16,2024-01-02 23:42:16,NaT,"[44.50781, -88.0221]",None,AIS
1795957,190,0,366893910,2024-01-02 23:43:25,2024-01-02 23:43:25,NaT,"[44.50858, -88.02165]",None,AIS


In [35]:
path2, probability2, log_probs2 = viterbi_model.viterbi(AIS_sequence2, RF_sequence2)

In [36]:
RF_M_indexes_pred_path2 = [(i, state) for i, state in enumerate(path2) if state in {"RF", "M"}]
RF_M_indexes_real_path2 = [(i, state) for i, state in enumerate(sequence2["State"]) if state in {"RF", "M"}]
print(f"The indexes of RF and/or M of predicted path are: {RF_M_indexes_pred_path2}, \
      \nThe indexes of RF and/or M of the real path are: {RF_M_indexes_real_path2}")

The indexes of RF and/or M of predicted path are: [(39, 'RF'), (101, 'RF'), (165, 'RF'), (228, 'RF'), (331, 'RF'), (527, 'RF'), (735, 'RF'), (892, 'RF'), (1052, 'RF'), (1223, 'RF'), (1385, 'RF'), (1581, 'RF'), (1776, 'RF'), (1980, 'RF')],       
The indexes of RF and/or M of the real path are: [(39, 'RF'), (101, 'RF'), (165, 'RF'), (228, 'RF'), (331, 'RF'), (527, 'RF'), (735, 'RF'), (892, 'RF'), (1052, 'RF'), (1223, 'RF'), (1385, 'RF'), (1581, 'RF'), (1776, 'RF'), (1980, 'RF')]


In [ ]:
def create_prediction_dataframe(path, AIS_sequence, RF_sequence):    
    results = []
    RF_M_counter = 0 
    AIS_idx = -1
 
    for state in path:
        if state == "AIS":
            AIS_idx += 1
            results.append({
                "State_pred": state,
                "AIS_Timestamp_pred": AIS_sequence[AIS_idx][0],
                "RF_Timestamp_pred": pd.NaT,
            })
        elif state == "RF":
            results.append({
                "State_pred": state,
                "AIS_Timestamp_pred": pd.NaT,
                "RF_Timestamp_pred": RF_sequence[RF_M_counter][0],
            })
            RF_M_counter += 1
        elif state == "M":
            AIS_idx += 1
            results.append({
                "State_pred": state,
                "AIS_Timestamp_pred": AIS_sequence[AIS_idx][0],
                "RF_Timestamp_pred": RF_sequence[RF_M_counter][0],
            })
            RF_M_counter += 1

    return pd.DataFrame(results)    

In [ ]:
def merge_predictions_and_actuals(df_pred, df_actual):
    df_pred = df_pred.copy()
    df_actual = df_actual.copy()

    # Filter begin/end uit actuals
    df_actual = df_actual[~df_actual['State'].isin(['begin', 'end'])][["State", "AIS_Timestamp", "RF_Timestamp"]]

    # Maak merge_keys
    df_pred['merge_key'] = df_pred['AIS_Timestamp_pred'].fillna(df_pred['RF_Timestamp_pred'])
    df_actual['merge_key'] = df_actual['AIS_Timestamp'].fillna(df_actual['RF_Timestamp'])

    # Merge
    merged = pd.merge(
        df_pred,
        df_actual,
        on='merge_key',
        how='outer',
        suffixes=('_pred', '_actual')
    ).drop(columns=['merge_key'])

    # Sorteren
    merged['sort_key'] = merged['AIS_Timestamp_pred'].fillna(merged['RF_Timestamp_pred'])
    merged = merged.sort_values('sort_key').drop(columns='sort_key')
    merged = merged.reset_index(drop=True)

    # Vul ontbrekende voorspellingen aan met '-'
    missing_pred = merged['State_pred'].isna()
    pred_cols = ['State_pred', 'AIS_Timestamp_pred', 'RF_Timestamp_pred']
    merged.loc[missing_pred, pred_cols] = '-'

    # Vul ontbrekende actuele data aan met '-'
    missing_actual = merged['State'].isna()
    actual_cols = ['State', 'AIS_Timestamp', 'RF_Timestamp']
    merged.loc[missing_actual, actual_cols] = '-'

    # Scheidingskolom
    merged.insert(3, "", "|")

    # Voeg indexkolommen toe (start met alles op NaN)
    merged.insert(0, "index_pred", "-")
    merged.insert(5, "index", "-")

    # Tel bij via counters
    pred_counter = 1
    actual_counter = 1

    for i, row in merged.iterrows():
        if row['State_pred'] != '-':
            merged.at[i, 'index_pred'] = pred_counter
            pred_counter += 1
        if row['State'] != '-':
            merged.at[i, 'index'] = actual_counter
            actual_counter += 1

    # Begin- en eindrijen toevoegen
    begin_row = {
        'index_pred': 0,
        'State_pred': 'begin',
        'AIS_Timestamp_pred': pd.NaT,
        'RF_Timestamp_pred': pd.NaT,
        '': '|',
        'index': 0,
        'State': 'begin',
        'AIS_Timestamp': pd.NaT,
        'RF_Timestamp': pd.NaT,  
    }

    end_row = {
        'index_pred': pred_counter,
        'State_pred': 'end',
        'AIS_Timestamp_pred': pd.NaT,
        'RF_Timestamp_pred': pd.NaT,
        '': '|',
        'index': actual_counter,
        'State': 'end',
        'AIS_Timestamp': pd.NaT,
        'RF_Timestamp': pd.NaT,
    }

    merged = pd.concat(
        [pd.DataFrame([begin_row]), merged, pd.DataFrame([end_row])],
        ignore_index=True
    )

    return merged


In [ ]:
df_pred = create_prediction_dataframe(path2, AIS_sequence2, RF_sequence2)

merge_predictions_and_actuals(df_pred, sequence2)

,index_pred,State_pred,AIS_Timestamp_pred,RF_Timestamp_pred,,index,State,AIS_Timestamp,RF_Timestamp
0,0,begin,NaN,NaN,|,0,begin,NaN,NaN
1,1,AIS,2024-01-01 02:32:02,NaT,|,1,AIS,2024-01-01 02:32:02,NaT
2,2,AIS,2024-01-01 02:38:32,NaT,|,2,AIS,2024-01-01 02:38:32,NaT
3,3,AIS,2024-01-01 02:42:02,NaT,|,3,AIS,2024-01-01 02:42:02,NaT
4,4,AIS,2024-01-01 02:43:52,NaT,|,4,AIS,2024-01-01 02:43:52,NaT
...,...,...,...,...,...,...,...,...,...
1117,1117,AIS,2024-01-02 00:28:53,NaT,|,1117,AIS,2024-01-02 00:28:53,NaT
1118,1118,AIS,2024-01-02 00:31:53,NaT,|,1118,AIS,2024-01-02 00:31:53,NaT
1119,1119,AIS,2024-01-02 00:38:32,NaT,|,1119,AIS,2024-01-02 00:38:32,NaT
1120,1120,AIS,2024-01-02 00:44:21,NaT,|,1120,AIS,2024-01-02 00:44:21,NaT


In [ ]:
from scripts.PHMM.viterbi_algorithm.forward import *

begin = BeginState()
AIS = AISState()
RF = RFState()
M = MState()
end = EndState()

states = [begin, AIS, RF, M, end]
forward_model = PHMM(states, end)

In [ ]:
# stap 1, verkrijg alleen RF data
RF_signals_df = PHMM_df[PHMM_df["RF"].notna()].copy()
RF_signals_df["RF_index_in_route"] = RF_signals_df.groupby("UniqueRouteID").cumcount() + 1

In [ ]:
RF_signals_df # samples cross validation samples 

,UniqueRouteID,SubRouteID,MMSI,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,RF_index_in_route
58,1,0,209136000,2024-01-08 23:07:44,2024-01-08 23:07:44,2024-01-08 23:08:37,"[28.56874, -89.66455]","[28.585413113999813, -89.66683609570418]",M,1
201,1,0,209136000,2024-01-09 02:07:00,NaT,2024-01-09 02:07:00,None,"[28.895653373648273, -89.4373688940996]",RF,2
355,1,0,209136000,2024-01-09 05:08:47,NaT,2024-01-09 05:08:47,None,"[29.235375467693093, -89.31853672724804]",RF,3
422,1,0,209136000,2024-01-09 08:26:02,NaT,2024-01-09 08:26:02,None,"[29.23928012791552, -89.27971225537458]",RF,4
484,1,0,209136000,2024-01-09 11:40:10,NaT,2024-01-09 11:40:10,None,"[29.247239827713063, -89.29967225633574]",RF,5
...,...,...,...,...,...,...,...,...,...,...
4405686,14564,539,368099310,2024-01-29 05:08:47,NaT,2024-01-29 05:08:47,None,"[28.57989782770889, -89.15286991480984]",RF,3
4405826,14567,550,366014000,2024-01-25 05:08:47,NaT,2024-01-25 05:08:47,None,"[28.677629487969426, -88.28475164952638]",RF,1
4405852,14569,556,366014000,2024-01-25 08:25:13,2024-01-25 08:25:13,2024-01-25 08:26:02,"[28.77424, -88.42494]","[28.782059969846472, -88.40670273757524]",M,1
4405985,14569,556,366014000,2024-01-25 11:40:10,NaT,2024-01-25 11:40:10,None,"[28.709489862140856, -88.36347282520185]",RF,2


In [ ]:
PHMM_df

,UniqueRouteID,SubRouteID,MMSI,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
0,0,0,209087000,2024-01-05 20:06:08,NaT,NaT,None,None,begin
1,0,0,209087000,2024-01-05 20:06:09,2024-01-05 20:06:09,NaT,"[25.05721, -95.32958]",None,AIS
2,0,0,209087000,2024-01-05 20:09:04,2024-01-05 20:09:04,NaT,"[25.068, -95.31942]",None,AIS
3,0,0,209087000,2024-01-05 20:11:21,2024-01-05 20:11:21,NaT,"[25.07645, -95.31147]",None,AIS
4,0,0,209087000,2024-01-05 20:12:33,2024-01-05 20:12:33,NaT,"[25.0809, -95.30727]",None,AIS
...,...,...,...,...,...,...,...,...,...
4406108,14571,562,366014000,2024-01-25 23:54:18,2024-01-25 23:54:18,NaT,"[28.74495, -88.39711]",None,AIS
4406109,14571,562,366014000,2024-01-25 23:55:38,2024-01-25 23:55:38,NaT,"[28.74411, -88.39585]",None,AIS
4406110,14571,562,366014000,2024-01-26 00:04:26,2024-01-26 00:04:26,NaT,"[28.73821, -88.38714]",None,AIS
4406111,14571,562,366014000,2024-01-26 00:06:46,2024-01-26 00:06:46,NaT,"[28.73672, -88.3849]",None,AIS


In [ ]:
# 0.99 van normaal verdeling genereerde RF punten 

In [ ]:
# stap 2, preselectie van mogelijke AIS sequenties op basis van:
    # tijdsvenster (+- 5 minuten)
    # ruimtelijke afstand (2 km van een van de AIS punten)

# stap 3, forward algorithme draaien op:
    # elke combinatie van het RF signaal + kandidaat AIS sequentie
    # sla deze allemaal op in een dataframe + ook de adjusted forward score (forward / len(AIS_seq))
    
candidates = []
distance_threshold = 2
time_marge = pd.Timedelta(minutes = 5) # deze 10 minuten 

for _, RF_signal in RF_signals_sample.iterrows():
    RF_datetime = RF_signal["RF_Timestamp"]
    RF_coordinates = RF_signal["RF"]
    RF_seq_id = RF_signal["UniqueRouteID"]
    RF_signal_id = RF_signal["RF_index_in_route"]
    
    best_score = -np.inf
    best_candidate = None
    best_adjusted_score = None
    
    for AIS_seq_id, AIS_seq in PHMM_df.groupby("UniqueRouteID"):
        # Check if RF signal within distance threshold
        AIS_coordinates = [coord for coord in AIS_seq["AIS"] if coord is not None]
        RF_signal_within_radius = any(haversine(RF_coordinates, coord) <= distance_threshold for coord in AIS_coordinates)
        
        # Check if RF signal within time bounds
        min_AIS_datetime = AIS_seq["AIS_Timestamp"].min() - time_marge
        max_AIS_datetime = AIS_seq["AIS_Timestamp"].max() + time_marge
        RF_signal_within_time = min_AIS_datetime <= RF_datetime <= max_AIS_datetime

        if RF_signal_within_radius and RF_signal_within_time:
            # Make sequenties 
            AIS_seq = [(datetime, coordinates)
                    for datetime, coordinates in zip(AIS_seq["AIS_Timestamp"], AIS_seq["AIS"])
                    if pd.notna(datetime) and coordinates is not None]
            RF_seq = [(RF_datetime, RF_coordinates)]
        
            # Forward algorithm 
            forward_score = forward_model.forward(AIS_seq, RF_seq)
            adjusted_forward_score = forward_score/len(AIS_seq)
            
            is_true_match = RF_seq_id == AIS_seq_id
            
            candidates.append({
                "RF_signal_ID": RF_signal_id,
                "RF_seq_id": RF_seq_id,
                "AIS_seq_ID": AIS_seq_id,
                "Forward_score": forward_score,
                "Adjusted_Forward_score": adjusted_forward_score, 
                "y_true": is_true_match
            })

KeyError: 'speed_kph'

In [ ]:
forward_results

,RF_signal_ID,RF_seq_id,AIS_seq_ID,Forward_score,Adjusted_Forward_score,y_true
0,1,10197,10197,-17.500984,-0.040325,True
1,9,8724,8724,-41.951284,-0.029987,True
2,8,1345,544,-67.860931,-0.028346,False
3,8,1345,1286,-122.587214,-0.078784,False
4,8,1345,1345,-44.018018,-0.029823,True
...,...,...,...,...,...,...
246,12,997,181,-475.946815,-0.065802,False
247,12,997,187,-361.885506,-0.051280,False
248,12,997,941,-86.619164,-0.031452,False
249,12,997,997,-86.907839,-0.028017,True


In [ ]:
# stap 4, bepaal de beste forward algorithme (hoger, minder negatieve waarde betekent betere alignment)
forward_results = pd.DataFrame(candidates)

idx = forward_results.groupby("RF_signal_ID")["Adjusted_Forward_score"].idxmax()
best_predictions = forward_results.loc[idx].reset_index(drop=True)          
                      
best_predictions.head(5)

,RF_signal_ID,RF_seq_id,AIS_seq_ID,Forward_score,Adjusted_Forward_score,y_true,y_pred
0,1,6329,6329,-61.667405,-0.028484,True,True
1,2,13767,13767,-20.241849,-0.037209,True,True
2,3,5991,5991,-51.136893,-0.029713,True,True
3,4,1811,2093,-172.236731,-0.026749,False,True
4,5,11957,11957,-47.005254,-0.029769,True,True


In [ ]:
# stap 5, run viterbi algorithme op de beste sequentie om zo te bepalen wat de optimale positie is van de RF signaal in de AIS sequentie 

Dit mag later verwijderd worden 

In [ ]:
# time_marge = 5
# generated_times = generate_RF_revisit_times()
# rows = []

# for (mmsi, route_id), group in processed_df.groupby(["MMSI", "route_id"]):
#     group = group.sort_values("BaseDateTime")
#     group["Matched"] = False
    
#     # Generate RF datetimes
#     start_time = group["BaseDateTime"].iloc[0] - timedelta(minutes = time_marge)
#     end_time = group["BaseDateTime"].iloc[-1] + timedelta(minutes = time_marge)
#     unique_dates = group['BaseDateTime'].dt.normalize().unique()
    
#     selected_RF_datetimes = sorted([
#         datetime.combine(pd.to_datetime(date).date(), RF_time.time())
#         for date in unique_dates
#         for RF_time in generated_times
#         if start_time <= datetime.combine(pd.to_datetime(date).date(), RF_time.time()) <= end_time
#     ])
    
#     error_data = generate_distance_bearing(selected_RF_datetimes)
    
#     for RF_datetime in selected_RF_datetimes:        
#         # Find closest AIS information 
#         closest_time_idx = (group["BaseDateTime"] - RF_datetime).abs().idxmin()
#         closest_AIS_time = group.loc[closest_time_idx, "BaseDateTime"]
#         closest_AIS_lat = group.loc[closest_time_idx, "LAT"]
#         closest_AIS_lon = group.loc[closest_time_idx, "LON"]
#         closest_AIS_heading = group.loc[closest_time_idx, "Heading"]
        
#         # Calculate the coordinates for RF signal
#         distance, bearing_noise = error_data[RF_datetime]
#         synthetic_RF_lat, synthetic_RF_lon = compute_coordinates(closest_AIS_lat, closest_AIS_lon, distance, closest_AIS_heading, bearing_noise)
        
#         # Calculate the difference in distance and time
#         distance_diff = haversine((closest_AIS_lat, closest_AIS_lon), (synthetic_RF_lat, synthetic_RF_lon))
#         time_diff = abs((RF_datetime - closest_AIS_time).total_seconds())
        
#         # Calculate the change of a match or a RF signal only
#         prob_RF = AIS_RF_probability(time_diff, distance_diff)
#         prob_match = match_probability(time_diff, distance_diff)
        
#         if prob_match > prob_RF:
#             state = "M"
#             group.at[closest_time_idx, "Matched"] = True 
#         else:
#             state = "RF"

#         rows.append({
#             "SubRouteID": route_id,
#             "MMSI": mmsi,
#             "Timestamp": closest_AIS_time if state == "M" else RF_datetime,
#             "AIS_Timestamp": closest_AIS_time if state == "M" else pd.NaT,
#             "RF_Timestamp": RF_datetime,
#             "AIS": [closest_AIS_lat, closest_AIS_lon] if state == "M" else None,
#             "RF": [synthetic_RF_lat, synthetic_RF_lon],
#             "State": state
#         })
        
#     unmatched = group[~group["Matched"]]
#     for i, row in unmatched.iterrows():
#         rows.append({
#             "SubRouteID": route_id,
#             "MMSI": mmsi,
#             "Timestamp": row["BaseDateTime"],
#             "AIS_Timestamp": row["BaseDateTime"],
#             "RF_Timestamp": pd.NaT,
#             "AIS": [row["LAT"], row["LON"]],
#             "RF": None,
#             "State": "AIS"
#         })
        
# PHMM_df = pd.DataFrame(rows).sort_values(["MMSI", "SubRouteID", "Timestamp"])
# PHMM_df.insert(0, "UniqueRouteID", PHMM_df.groupby(["SubRouteID", "MMSI"]).ngroup())

In [ ]:
# begin_end_rows = []

# for (id, route_id, mmsi), group in PHMM_df.groupby(["UniqueRouteID", "SubRouteID", "MMSI"]):
#     time_begin_state = group["Timestamp"].min() - timedelta(seconds = 1)
#     time_end_state = group["Timestamp"].max() + timedelta(seconds = 1)
    
#     begin_row = {
#         "UniqueRouteID": id,
#         "SubRouteID": route_id,
#         "MMSI": mmsi,
#         "Timestamp": time_begin_state,
#         "AIS_Timestamp": pd.NaT,
#         "RF_Timestamp": pd.NaT,
#         "AIS": None,
#         "RF": None,
#         "State": "begin"
#     }
    
#     end_row = {
#         "UniqueRouteID": id,
#         "SubRouteID": route_id,
#         "MMSI": mmsi,
#         "Timestamp": time_end_state,
#         "AIS_Timestamp": pd.NaT,
#         "RF_Timestamp": pd.NaT,
#         "AIS": None,
#         "RF": None,
#         "State": "end"
#     }
    
#     begin_end_rows.extend([begin_row, end_row])

# PHMM_df = pd.concat([PHMM_df, pd.DataFrame(begin_end_rows)], ignore_index=True)
# PHMM_df = PHMM_df.sort_values(["UniqueRouteID", "Timestamp"]).reset_index(drop=True)

In [ ]:
# time_marge = 30
# rows = []

# for (mmsi, route_id), group in processed_df.groupby(["MMSI", "route_id"]):
#     group = group.sort_values("BaseDateTime")
#     group["Matched"] = False
    
#     start_time = group["BaseDateTime"].iloc[0] - timedelta(minutes = time_marge)
#     end_time = group["BaseDateTime"].iloc[-1] + timedelta(minutes = time_marge)
#     unique_dates = group['BaseDateTime'].dt.normalize().unique()
    
#     selected_RF_times = sorted([
#         datetime.combine(pd.to_datetime(date).date(), RF_time.time())
#         for date in unique_dates
#         for RF_time in generated_times
#         if start_time <= datetime.combine(pd.to_datetime(date).date(), RF_time.time()) <= end_time
#     ])
    
#     error_data = generate_distance_bearing(selected_RF_times)
    
#     for RF_datetime in selected_RF_times:
#         closest_id = (group["BaseDateTime"] - RF_datetime).abs().idxmin()
#         closest_row = group.loc[closest_id]
#         id = group.index.get_loc(closest_id)
        
#         lat = closest_row["LAT"]
#         long = closest_row["LON"]
#         heading = closest_row["Heading"]
#         distance, bearing = error_data[RF_datetime]
        
#         RF_lat, RF_long = compute_coordinates(lat, long, distance, heading, bearing)
        
#         closest_AIS_time = closest_row["BaseDateTime"]
                
#         distance_diff_km = haversine((lat, long), (RF_lat, RF_long), unit=Unit.KILOMETERS)
#         time_diff_sec = abs((RF_datetime - closest_AIS_time).total_seconds())
        
#         if RF_datetime < closest_AIS_time and id > 0:
#             closest_row_prev = group.iloc[id - 1]
#             prev_AIS_time = closest_row_prev['BaseDateTime']
#             lat_prev = closest_row_prev["LAT"]
#             long_prev = closest_row_prev["LON"]
            
#             distance_diff_km_prev = haversine((lat_prev, long_prev), (RF_lat, RF_long), unit=Unit.KILOMETERS)
#             time_diff_sec_prev = abs((RF_datetime - prev_AIS_time).total_seconds())
            
#             prob_AIS_RF = AIS_RF_probability(time_diff_sec_prev, distance_diff_km_prev)
#         else:
#             prob_AIS_RF = AIS_RF_probability(time_diff_sec, distance_diff_km)
        
#         prob_match = match_probability(time_diff_sec, distance_diff_km)
        
#         if prob_match > prob_AIS_RF:
#             state = "M"
#             group.at[closest_id, "Matched"] = True 
#         else:
#             state = "RF"
        
#         rows.append({
#             "SubRouteID": route_id,
#             "MMSI": mmsi,
#             "Timestamp": closest_AIS_time if state == "M" else RF_datetime,
#             "AIS_Timestamp": closest_AIS_time if state == "M" else pd.NaT,
#             "RF_Timestamp": RF_datetime,
#             "AIS": [lat, long] if state == "M" else None,
#             "RF": [RF_lat, RF_long],
#             "State": state
#         })
        
#     unmatched = group[~group["Matched"]]
#     for i, row in unmatched.iterrows():
#         rows.append({
#             "SubRouteID": route_id,
#             "MMSI": mmsi,
#             "Timestamp": row["BaseDateTime"],
#             "AIS_Timestamp": row["BaseDateTime"],
#             "RF_Timestamp": pd.NaT,
#             "AIS": [row["LAT"], row["LON"]],
#             "RF": None,
#             "State": "AIS"
#         })
        
#     time_begin_state = group["BaseDateTime"].min() - timedelta(seconds = 1)
#     time_end_state = group["BaseDateTime"].max() + timedelta(seconds = 1)
    
#     begin_row = {
#         "SubRouteID": route_id,
#         "MMSI": mmsi,
#         "Timestamp": time_begin_state,
#         "AIS_Timestamp": pd.NaT,
#         "RF_Timestamp": pd.NaT,
#         "AIS": None,
#         "RF": None,
#         "State": "begin"
#     }
    
#     end_row = {
#         "SubRouteID": route_id,
#         "MMSI": mmsi,
#         "Timestamp": time_end_state,
#         "AIS_Timestamp": pd.NaT,
#         "RF_Timestamp": pd.NaT,
#         "AIS": None,
#         "RF": None,
#         "State": "end"
#     }
    
#     rows.extend([begin_row, end_row])
    
# PHMM_df = pd.DataFrame(rows).sort_values(["MMSI", "SubRouteID", "Timestamp"])
# PHMM_df.insert(0, "UniqueRouteID", PHMM_df.groupby(["SubRouteID", "MMSI"]).ngroup())

In [ ]:
# def merge_predictions_and_actuals(df_pred, df_actual):
#     df_actual = df_actual[~df_actual['State'].isin(['begin', 'end'])][["State", "AIS_Timestamp", "RF_Timestamp"]]
    
#     df_pred['merge_key'] = df_pred['AIS_Timestamp_pred'].fillna(df_pred['RF_Timestamp_pred'])
#     df_actual['merge_key'] = df_actual['AIS_Timestamp'].fillna(df_actual['RF_Timestamp'])
    
#     merged = pd.merge(
#         df_pred,
#         df_actual,
#         on='merge_key',
#         how='outer',
#         suffixes=('_pred', '_actual')
#     ).drop(columns=['merge_key'])

#     merged['sort_key'] = merged['AIS_Timestamp_pred'].fillna(merged['RF_Timestamp_pred'])
#     merged = merged.sort_values('sort_key').drop(columns='sort_key')
#     merged = merged.reset_index(drop=True)
    
#     # Vul ontbrekende voorspellingen aan met '-'
#     missing_pred = merged['State_pred'].isna()
#     pred_cols = ['State_pred', 'AIS_Timestamp_pred', 'RF_Timestamp_pred']
#     merged.loc[missing_pred, pred_cols] = '-'
    
#     # Vul ontbrekende actuele data aan met '-'
#     missing_actual = merged['State'].isna()
#     actual_cols = ['State', 'AIS_Timestamp', 'RF_Timestamp']
#     merged.loc[missing_actual, actual_cols] = '-'
    
#     # Voeg scheidingskolom toe
#     merged.insert(3, "", "|")
    
#     # Begin- en eindrij
#     begin_row = {
#         'State_pred': 'begin',
#         'AIS_Timestamp_pred': pd.NaT,
#         'RF_Timestamp_pred': pd.NaT,
#         '': '|',
#         'State': 'begin',
#         'AIS_Timestamp': pd.NaT,
#         'RF_Timestamp': pd.NaT,
#     }

#     end_row = {
#         'State_pred': 'end',
#         'AIS_Timestamp_pred': pd.NaT,
#         'RF_Timestamp_pred': pd.NaT,
#         '': '|',
#         'State': 'end',
#         'AIS_Timestamp': pd.NaT,
#         'RF_Timestamp': pd.NaT,
#     }
    
#     # Voeg toe aan begin en eind
#     merged = pd.concat(
#         [pd.DataFrame([begin_row]), merged, pd.DataFrame([end_row])],
#         ignore_index=True
#     )

#     return merged